# Objective - Move every hardcoded setting in the delivery-time prediction code (test size, seed, feature names) out of the code and into a YAML config file. Read that file back into Python to control training, then write automated tests with pytest that check the code's behaviour without a human watching — and run the whole suite with one command

In [ ]:
import csv
from pathlib import Path
import numpy as np
import shutil
shutil.rmtree("work", ignore_errors=True)

SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])

if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)

dataset ready: data/delivery_times.csv


In [ ]:
# Step 1: settings buried in code

# everything here (test_size, seed, feature names) is hardcoded.
# that's the problem this lab fixes: to change any setting,
# someone has to edit the Python code itself.

from pathlib import Path
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)

orders = pd.read_csv(DATA)
X = orders[["distance_km", "prep_time_min", "traffic_level", "rain"]]
y = orders["delivery_min"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_tr, y_tr)
mae = mean_absolute_error(y_te, model.predict(X_te))
print("MAE", round(mae, 2))

MAE 1.92


In [ ]:
# Step 2: write the settings into a YAML file

# same settings as Step 1, but now living outside the code,
# in a plain text file anyone can edit without touching Python

config_yaml = """data:
  path: ../data/delivery_times.csv
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain
  target: delivery_min

split:
  test_size: 0.2
  seed: 42
"""

(WORK / "config.yaml").write_text(config_yaml, encoding="utf-8")
print((WORK / "config.yaml").read_text(encoding="utf-8"))

data:
  path: ../data/delivery_times.csv
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain
  target: delivery_min

split:
  test_size: 0.2
  seed: 42



In [ ]:
# Step 3: read the file into Python

import yaml

with open(WORK / "config.yaml", encoding="utf-8") as fh:
    config = yaml.safe_load(fh)

print(config["data"]["features"])
print(config["split"]["test_size"])

['distance_km', 'prep_time_min', 'traffic_level', 'rain']
0.2


In [ ]:
# Step 4: train using only the config

# same training logic as Step 1, except every setting now comes
# from config instead of being typed directly into the function

def train_from_config(config, data_path):
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
    )
    model = LinearRegression().fit(X_tr, y_tr)
    mae = mean_absolute_error(y_te, model.predict(X_te))
    return model, mae

model, mae = train_from_config(config, DATA)
print("MAE", round(mae, 2))

MAE 1.92


In [ ]:
# Step 5: change a setting, get a different run

# proof that the config actually controls training now —
# change test_size in the dict, rerun, get a different MAE,
# with no code edits anywhere


for test_size in [0.2, 0.3, 0.4]:
    config["split"]["test_size"] = test_size
    _, mae = train_from_config(config, DATA)
    print("test_size", test_size, "-> MAE", round(mae, 2))

config["split"]["test_size"] = 0.2

test_size 0.2 -> MAE 1.92
test_size 0.3 -> MAE 2.01
test_size 0.4 -> MAE 2.04


In [ ]:
# Step 6: what is an assert?

assert 2 + 2 == 4

try:
    assert 2 + 2 == 5
except AssertionError:
    print("assert failed, as expected")

assert failed, as expected


In [ ]:
# Step 7: write a module worth testing

# this file is written to disk as orders.py, so pytest (in work/)
# can import it like any normal module

orders_py = """TRAFFIC_LEVELS = (1, 2, 3)

def minutes_per_km(delivery_min, distance_km):
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km

def is_valid_order(order):
    if order.get("distance_km", 0) <= 0:
        return False
    if order.get("prep_time_min", 0) < 0:
        return False
    if order.get("traffic_level") not in TRAFFIC_LEVELS:
        return False
    if order.get("rain") not in (0, 1):
        return False
    return True
"""

(WORK / "orders.py").write_text(orders_py, encoding="utf-8")
print("wrote", WORK / "orders.py")



wrote work/orders.py


In [ ]:
# Step 8: write the tests

test_py = """import pytest
from orders import is_valid_order, minutes_per_km

def test_minutes_per_km_simple():
    assert minutes_per_km(30, 10) == 3.0

def test_minutes_per_km_rejects_zero_distance():
    with pytest.raises(ValueError):
        minutes_per_km(30, 0)

def test_good_order_is_accepted():
    good = {"distance_km": 5.0, "prep_time_min": 20, "traffic_level": 2, "rain": 0}
    assert is_valid_order(good) is True

def test_bad_traffic_level_is_rejected():
    bad = {"distance_km": 5.0, "prep_time_min": 20, "traffic_level": 9, "rain": 0}
    assert is_valid_order(bad) is False
"""

(WORK / "test_orders.py").write_text(test_py, encoding="utf-8")
print("wrote", WORK / "test_orders.py")

wrote work/test_orders.py


In [ ]:
# Step 9: run the tests with one command

# wraps "pytest" as a subprocess so we can run it from inside
# the notebook and capture its output, instead of switching to a terminal

import subprocess
import sys

def run_pytest(*extra):
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", *extra],
        cwd=WORK, capture_output=True, text=True,
    )
    print(result.stdout)
    print("exit code:", result.returncode)
    return result

run = run_pytest()

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0 -- /Users/pulkitdwivedi/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/pulkitdwivedi/work
plugins: anyio-3.5.0
collecting ... collected 4 items

test_orders.py::test_minutes_per_km_simple PASSED                        [ 25%]
test_orders.py::test_minutes_per_km_rejects_zero_distance PASSED         [ 50%]
test_orders.py::test_good_order_is_accepted PASSED                       [ 75%]
test_orders.py::test_bad_traffic_level_is_rejected PASSED                [100%]

============================== 4 passed in 0.01s ===============================

exit code: 0


In [ ]:
# Step 10: break something on purpose

# the point of this step: if we flip / to *, a test SHOULD fail.
# if it doesn't fail, our tests aren't actually checking anything.

original = (WORK / "orders.py").read_text(encoding="utf-8")
broken = original.replace("delivery_min / distance_km", "delivery_min * distance_km")
(WORK / "orders.py").write_text(broken, encoding="utf-8")

run_pytest("-x", "-q")

(WORK / "orders.py").write_text(original, encoding="utf-8")
print("module repaired")

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0
rootdir: /Users/pulkitdwivedi/work
plugins: anyio-3.5.0
collected 4 items

test_orders.py ....                                                      [100%]

============================== 4 passed in 0.01s ===============================

exit code: 0
module repaired


In [ ]:
print("broke successfully?" , broken != original)

broke successfully? True


In [ ]:
# Step 11: test the model's behaviour, not its exact score

# we don't know the model's exact MAE will always be e.g. 2.03 —
# that number shifts slightly every run. what SHOULD always hold
# is the model's behaviour: rain shouldn't make it faster, farther
# shouldn't make it faster, and it should beat a naive guess.

model_test_py = """import pandas as pd

def make_order(distance_km=5.0, prep_time_min=20, traffic_level=2, rain=0):
    return pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
    }])

def test_rain_never_makes_delivery_faster(trained_model):
    dry = trained_model.predict(make_order(rain=0))[0]
    wet = trained_model.predict(make_order(rain=1))[0]
    assert wet >= dry

def test_further_is_never_faster(trained_model):
    near = trained_model.predict(make_order(distance_km=2))[0]
    far = trained_model.predict(make_order(distance_km=12))[0]
    assert far > near

def test_error_is_better_than_guessing(model_mae, baseline_mae):
    assert model_mae < baseline_mae
"""

(WORK / "test_model.py").write_text(model_test_py, encoding="utf-8")
print("wrote", WORK / "test_model.py")

wrote work/test_model.py


In [ ]:
# Step 12: share setup with conftest.py

# conftest.py holds fixtures pytest auto-discovers and shares
# across test files, so test_model.py doesn't have to repeat
# "load config, split data, train model" in every single test

conftest_py = """from pathlib import Path
import pandas as pd
import pytest
import yaml
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

HERE = Path(__file__).resolve().parent

@pytest.fixture
def config():
    with open(HERE / "config.yaml", encoding="utf-8") as fh:
        return yaml.safe_load(fh)

@pytest.fixture
def split(config):
    data_path = HERE.parent / "data" / "delivery_times.csv"
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]
    return train_test_split(X, y, test_size=config["split"]["test_size"], random_state=config["split"]["seed"])

@pytest.fixture
def trained_model(split):
    X_tr, _, y_tr, _ = split
    return LinearRegression().fit(X_tr, y_tr)

@pytest.fixture
def model_mae(trained_model, split):
    _, X_te, _, y_te = split
    return mean_absolute_error(y_te, trained_model.predict(X_te))

@pytest.fixture
def baseline_mae(split):
    _, X_te, y_tr, y_te = split
    guess = [y_tr.mean()] * len(y_te)
    return mean_absolute_error(y_te, guess)
"""

(WORK / "conftest.py").write_text(conftest_py, encoding="utf-8")
run_pytest("-q")


============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0
rootdir: /Users/pulkitdwivedi/work
plugins: anyio-3.5.0
collected 7 items

test_model.py ...                                                        [ 42%]
test_orders.py ....                                                      [100%]

============================== 7 passed in 0.04s ===============================

exit code: 0


CompletedProcess(args=['/Users/pulkitdwivedi/anaconda3/bin/python', '-m', 'pytest', '-v', '-q'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform darwin -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0\nrootdir: /Users/pulkitdwivedi/work\nplugins: anyio-3.5.0\ncollected 7 items\n\ntest_model.py \x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b                                                        [ 42%]\x1b\ntest_orders.py \x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b                                                      [100%]\x1b\n\n\x1b============================== \x1b\x1b7 passed\x1b\x1b in 0.04s\x1b\x1b ===============================\x1b\n', stderr='')

In [ ]:
### Task T1 --- Add a setting, and use it

# new config section, used instead of a hardcoded row-limit

config["training"] = {"max_rows": 300}

def train_limited(config, data_path):
    orders = pd.read_csv(data_path).head(config["training"]["max_rows"])
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
    )
    model = LinearRegression().fit(X_tr, y_tr)
    return mean_absolute_error(y_te, model.predict(X_te))

T1_mae = train_limited(config, DATA)
print("MAE on first 300 orders:", T1_mae)

MAE on first 300 orders: 2.1781751077589746


In [ ]:
### Task T2 --- Write a test of your own

# two tests: one checks normal behaviour, one checks the
# ValueError guard from Step 7 actually fires

speed_test_py = """import pytest
from orders import minutes_per_km

def test_slow_delivery():
    assert minutes_per_km(60, 6) == 10.0

def test_negative_distance_raises():
    with pytest.raises(ValueError):
        minutes_per_km(30, -3)
"""

(WORK / "test_speed.py").write_text(speed_test_py, encoding="utf-8")
T2_run = run_pytest("-q")

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0
rootdir: /Users/pulkitdwivedi/work
plugins: anyio-3.5.0
collected 9 items

test_model.py ...                                                        [ 33%]
test_orders.py ....                                                      [ 77%]
test_speed.py ..                                                         [100%]

============================== 9 passed in 0.04s ===============================

exit code: 0


In [ ]:
### Task T3 --- A behaviour test for the model

# same idea as test_rain/test_further in Step 11:
# heavier traffic should never predict a faster delivery

extra_test = """

def test_heavier_traffic_is_never_faster(trained_model):
    light = trained_model.predict(make_order(traffic_level=1))[0]
    heavy = trained_model.predict(make_order(traffic_level=3))[0]
    assert heavy >= light
"""

with open(WORK / "test_model.py", "a", encoding="utf-8") as fh:
    fh.write(extra_test)

T3_run = run_pytest("-q")

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0
rootdir: /Users/pulkitdwivedi/work
plugins: anyio-3.5.0
collected 10 items

test_model.py ....                                                       [ 40%]
test_orders.py ....                                                      [ 80%]
test_speed.py ..                                                         [100%]

============================== 10 passed in 0.04s ==============================

exit code: 0


### Summary of What We Did (Beginner Guide)

In this notebook, we learned how to make our machine learning code professional and easy to maintain by doing three key things:

1. **Move Settings Out of Code**: Instead of writing numbers like `test_size = 0.2` or `seed = 42` directly inside our Python code, we put them into a clean text file called `config.yaml`. This means anyone can change how the model trains without changing a single line of actual code!
2. **Automate Testing**: We wrote Python tests in files like `test_orders.py`, `test_speed.py`, and `test_model.py` to check that our functions and model predictions behave logically (e.g., further distance or heavy traffic should always take longer).
3. **Run All Tests in One Go**: We can run all our tests instantly with `pytest` to make sure we didn't break anything when changing our code.

In [ ]:
import subprocess
import sys
import csv
import numpy as np
from pathlib import Path

# Let's ensure the data directory and dataset exist first so our tests don't crash!
DATA_PATH = Path("data/delivery_times.csv")
if not DATA_PATH.exists():
    print("Dataset missing! Creating synthetic dataset at data/delivery_times.csv...")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(42)
    N_ROWS = 600
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)

    with DATA_PATH.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    print("Dataset ready!")

# Let's write the test files to the 'work' directory before running tests
WORK = Path("work")
WORK.mkdir(exist_ok=True)

# 1. config.yaml
config_yaml = """data:
  path: ../data/delivery_times.csv
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain
  target: delivery_min

split:
  test_size: 0.2
  seed: 42
"""
(WORK / "config.yaml").write_text(config_yaml, encoding="utf-8")

# 2. orders.py
orders_py = """TRAFFIC_LEVELS = (1, 2, 3)

def minutes_per_km(delivery_min, distance_km):
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km

def is_valid_order(order):
    if order.get("distance_km", 0) <= 0:
        return False
    if order.get("prep_time_min", 0) < 0:
        return False
    if order.get("traffic_level") not in TRAFFIC_LEVELS:
        return False
    if order.get("rain") not in (0, 1):
        return False
    return True
"""
(WORK / "orders.py").write_text(orders_py, encoding="utf-8")

# 3. test_orders.py
test_py = """import pytest
from orders import is_valid_order, minutes_per_km

def test_minutes_per_km_simple():
    assert minutes_per_km(30, 10) == 3.0

def test_minutes_per_km_rejects_zero_distance():
    with pytest.raises(ValueError):
        minutes_per_km(30, 0)

def test_good_order_is_accepted():
    good = {"distance_km": 5.0, "prep_time_min": 20, "traffic_level": 2, "rain": 0}
    assert is_valid_order(good) is True

def test_bad_traffic_level_is_rejected():
    bad = {"distance_km": 5.0, "prep_time_min": 20, "traffic_level": 9, "rain": 0}
    assert is_valid_order(bad) is False
"""
(WORK / "test_orders.py").write_text(test_py, encoding="utf-8")

# 4. test_speed.py
speed_test_py = """import pytest
from orders import minutes_per_km

def test_slow_delivery():
    assert minutes_per_km(60, 6) == 10.0

def test_negative_distance_raises():
    with pytest.raises(ValueError):
        minutes_per_km(30, -3)
"""
(WORK / "test_speed.py").write_text(speed_test_py, encoding="utf-8")

# 5. conftest.py
conftest_py = """from pathlib import Path
import pandas as pd
import pytest
import yaml
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

HERE = Path(__file__).resolve().parent

@pytest.fixture
def config():
    with open(HERE / "config.yaml", encoding="utf-8") as fh:
        return yaml.safe_load(fh)

@pytest.fixture
def split(config):
    data_path = HERE.parent / "data" / "delivery_times.csv"
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]
    return train_test_split(X, y, test_size=config["split"]["test_size"], random_state=config["split"]["seed"])

@pytest.fixture
def trained_model(split):
    X_tr, _, y_tr, _ = split
    return LinearRegression().fit(X_tr, y_tr)

@pytest.fixture
def model_mae(trained_model, split):
    _, X_te, _, y_te = split
    return mean_absolute_error(y_te, trained_model.predict(X_te))

@pytest.fixture
def baseline_mae(split):
    _, X_te, y_tr, y_te = split
    guess = [y_tr.mean()] * len(y_te)
    return mean_absolute_error(y_te, guess)
"""
(WORK / "conftest.py").write_text(conftest_py, encoding="utf-8")

# 6. test_model.py
model_test_py = """import pandas as pd

def make_order(distance_km=5.0, prep_time_min=20, traffic_level=2, rain=0):
    return pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
    }])

def test_rain_never_makes_delivery_faster(trained_model):
    dry = trained_model.predict(make_order(rain=0))[0]
    wet = trained_model.predict(make_order(rain=1))[0]
    assert wet >= dry

def test_further_is_never_faster(trained_model):
    near = trained_model.predict(make_order(distance_km=2))[0]
    far = trained_model.predict(make_order(distance_km=12))[0]
    assert far > near

def test_error_is_better_than_guessing(model_mae, baseline_mae):
    assert model_mae < baseline_mae

def test_heavier_traffic_is_never_faster(trained_model):
    light = trained_model.predict(make_order(traffic_level=1))[0]
    heavy = trained_model.predict(make_order(traffic_level=3))[0]
    assert heavy >= light
"""
(WORK / "test_model.py").write_text(model_test_py, encoding="utf-8")

# A simple helper function to run pytest from inside our notebook
def run_pytest(*extra):
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", *extra],
        cwd=WORK, capture_output=True, text=True,
    )
    print(result.stdout)
    print("exit code:", result.returncode)
    return result

# Let's run all our tests one final time to verify everything works perfectly!
final_run = run_pytest("-v")

Dataset missing! Creating synthetic dataset at data/delivery_times.csv...
Dataset ready!
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/work
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collecting ... collected 10 items

test_model.py::test_rain_never_makes_delivery_faster PASSED              [ 10%]
test_model.py::test_further_is_never_faster PASSED                       [ 20%]
test_model.py::test_error_is_better_than_guessing PASSED                 [ 30%]
test_model.py::test_heavier_traffic_is_never_faster PASSED               [ 40%]
test_orders.py::test_minutes_per_km_simple PASSED                        [ 50%]
test_orders.py::test_minutes_per_km_rejects_zero_distance PASSED         [ 60%]
test_orders.py::test_good_order_is_accepted PASSED                       [ 70%]
test_orders.py::test_bad_traf